# Lesson 2.3 - Probability & Statistics (Titanic Dataset)

## Objectives
- Compute marginal and conditional probabilities.
- Analyze distributions and summary statistics.
- Map findings to risk-aware business decisions.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
TITANIC_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"


## Section A - Basic Stats


In [ ]:
try:
    titanic_df = pd.read_csv(TITANIC_URL)
except Exception as exc:
    raise RuntimeError("Titanic download failed") from exc

core_cols = ["survived", "sex", "pclass", "age", "fare"]
missing = [c for c in core_cols if c not in titanic_df.columns]
if missing:
    raise KeyError(f"Missing columns: {missing}")

print(titanic_df.shape)
display(titanic_df[core_cols].head())
display(titanic_df[["survived", "age", "fare"]].describe())


## Section B - Conditional Probability


In [ ]:
def conditional_survival(df, column, value):
    sub = df[df[column] == value]
    if len(sub) == 0:
        raise ValueError(f"No rows for {column}={value}")
    return float(sub["survived"].mean())

print("P(survived):", round(float(titanic_df["survived"].mean()), 4))
print("P(survived|female):", round(conditional_survival(titanic_df, "sex", "female"), 4))
print("P(survived|male):", round(conditional_survival(titanic_df, "sex", "male"), 4))
for c in sorted(titanic_df["pclass"].dropna().unique()):
    print(c, round(conditional_survival(titanic_df, "pclass", c), 4))


## Section C - Distribution Exploration


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(titanic_df["age"].dropna(), bins=25, kde=True, ax=axes[0]); axes[0].set_title("Age")
sns.histplot(titanic_df["fare"].dropna(), bins=25, kde=True, ax=axes[1]); axes[1].set_title("Fare")
plt.show()

print(titanic_df["age"].quantile([0.1,0.25,0.5,0.75,0.9]))
print(titanic_df["fare"].quantile([0.1,0.25,0.5,0.75,0.9]))


## Section D - Simple Risk View


In [ ]:
def toy_risk(row):
    score = 0.5
    score += 0.25 if row["sex"] == "male" else -0.15
    score += 0.2 if row["pclass"] == 3 else (-0.1 if row["pclass"] == 1 else 0.0)
    score += 0.1 if pd.notna(row["age"]) and row["age"] > 50 else 0.0
    return max(0.0, min(1.0, score))

risk_df = titanic_df.copy()
risk_df["toy_risk"] = risk_df.apply(toy_risk, axis=1)
print(risk_df.groupby(pd.cut(risk_df["toy_risk"], bins=[0,0.4,0.6,1.0]))["survived"].mean())


## Section E - Data Issues & Checks


In [ ]:
print(titanic_df[core_cols].isna().mean().sort_values(ascending=False))
clean_df = titanic_df.copy()
clean_df["age"] = clean_df["age"].fillna(clean_df["age"].median())
clean_df["fare"] = clean_df["fare"].fillna(clean_df["fare"].median())
print(clean_df[["age", "fare"]].isna().sum())


## Business Case Studies & Exceptions
- Segment-level conditional probabilities matter more than single global averages.
- Missing-value handling can shift risk estimates.
- Guard against zero-denominator conditional groups.


## Interview Questions & Answers
1. **Q:** Conditional probability formula?  
   **A:** `P(A|B)=P(A∩B)/P(B)`.
2. **Q:** Why class imbalance matters?  
   **A:** Base rates can mislead naive interpretation.
3. **Q:** Mean vs median with outliers?  
   **A:** Median is more robust.
4. **Q:** Why check missingness first?  
   **A:** Missing data can bias estimates.
5. **Q:** Independence definition?  
   **A:** `P(A∩B)=P(A)P(B)`.
6. **Q:** Why avoid causal claims here?  
   **A:** Observational data has confounding.
7. **Q:** Why quantiles useful?  
   **A:** Tail-risk understanding.
8. **Q:** Joint vs marginal?  
   **A:** Joint is combined events; marginal single event.
9. **Q:** How can probabilities support operations?  
   **A:** Prioritize actions by risk.
10. **Q:** Common conditional-probability pitfall?  
    **A:** Ignoring subgroup size/context.
